# Posterior ensemble

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Visualization in the latent space

#### Load the prior and posterior (latent space)

In [ ]:
import xarray as xr
import numpy as np

prior = xr.open_dataset(output_dir / "prior.nc")
posterior = xr.open_dataset(output_dir / "posterior-latent.nc")

nens = prior.sizes['ens']
nlatent = prior.sizes['latent_dim']

Z_prior    = prior['z'].to_numpy()
Z_analysis = posterior['z'].to_numpy()

In [ ]:
nlatent, nens

#### Visualization

In [ ]:
# Pick a few randor pairs
N = 6

rng = np.random.default_rng(42)

dims = rng.permutation(nlatent)[:2*N]

pairs = [
    (dims[2*k], dims[2*k + 1])
    for k in range(N)
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

# ------------------------------------------------------------
# Publication settings
# ------------------------------------------------------------

sns.set_theme(
    style="white",
    context="paper",
)

fig = plt.figure(
    figsize=(10, 6.8),
    dpi=300,
)

outer = fig.add_gridspec(
    2, 3,
    wspace=0.10,
    hspace=0.15,
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

for k, (i, j) in enumerate(pairs):

    row = k // 3
    col = k % 3

    inner = outer[row, col].subgridspec(
        2, 2,

        # Bigger marginals
        width_ratios=[3.5, 2.0],
        height_ratios=[2.0, 3.5],

        wspace=0.02,
        hspace=0.02,
    )

    ax = fig.add_subplot(inner[1, 0])

    ax_top = fig.add_subplot(
        inner[0, 0],
        sharex=ax,
    )

    ax_right = fig.add_subplot(
        inner[1, 1],
        sharey=ax,
    )

    # --------------------------------------------------------
    # Main scatter
    # --------------------------------------------------------

    ax.scatter(
        Z_prior[:, j],
        Z_prior[:, i],
        s=7,
        alpha=0.18,
        linewidth=0,
        label="Prior",
    )

    ax.scatter(
        Z_analysis[:, j],
        Z_analysis[:, i],
        s=7,
        alpha=0.35,
        linewidth=0,
        label="Posterior",
    )

    # --------------------------------------------------------
    # Common histogram ranges
    # --------------------------------------------------------

    x_all = np.concatenate([
        Z_prior[:, j],
        Z_analysis[:, j],
    ])

    y_all = np.concatenate([
        Z_prior[:, i],
        Z_analysis[:, i],
    ])

    x_bins = np.linspace(
        x_all.min(),
        x_all.max(),
        40,
    )

    y_bins = np.linspace(
        y_all.min(),
        y_all.max(),
        40,
    )

    # --------------------------------------------------------
    # Top histogram
    # --------------------------------------------------------

    ax_top.hist(
        Z_prior[:, j],
        bins=x_bins,
        density=True,
        alpha=0.35,
        linewidth=0,
    )

    ax_top.hist(
        Z_analysis[:, j],
        bins=x_bins,
        density=True,
        alpha=0.60,
        linewidth=0,
    )

    # --------------------------------------------------------
    # Right histogram
    # --------------------------------------------------------

    ax_right.hist(
        Z_prior[:, i],
        bins=y_bins,
        density=True,
        orientation="horizontal",
        alpha=0.35,
        linewidth=0,
    )

    ax_right.hist(
        Z_analysis[:, i],
        bins=y_bins,
        density=True,
        orientation="horizontal",
        alpha=0.60,
        linewidth=0,
    )

    # --------------------------------------------------------
    # Remove histogram axes
    # --------------------------------------------------------

    ax_top.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False,
    )

    ax_right.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False,
    )

    ax_top.set_ylabel("")
    ax_top.set_xlabel("")

    ax_right.set_ylabel("")
    ax_right.set_xlabel("")

    # --------------------------------------------------------
    # Main axes
    # --------------------------------------------------------

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(
        labelsize=8,
        direction="out",
        length=3,
    )

    # --------------------------------------------------------
    # Pair label
    # --------------------------------------------------------

    ax.text(
        0.04,
        0.94,
        rf"$Z_{{{i}}}$ vs $Z_{{{j}}}$",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
    )

    # --------------------------------------------------------
    # Panel label
    # --------------------------------------------------------

    ax.text(
        0.04,
        0.06,
        f"({chr(97 + k)})",
        transform=ax.transAxes,
        fontsize=10,
        #fontweight="bold",
    )

    # --------------------------------------------------------
    # Clean spines
    # --------------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax_top.spines["top"].set_visible(False)
    ax_top.spines["right"].set_visible(False)
    ax_top.spines["left"].set_visible(False)

    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    ax_right.spines["bottom"].set_visible(False)

# Legend
legend_handles = [
    Patch(
        facecolor=plt.rcParams["axes.prop_cycle"].by_key()["color"][0],
        alpha=0.35,
        label="Prior",
    ),
    Patch(
        facecolor=plt.rcParams["axes.prop_cycle"].by_key()["color"][1],
        alpha=0.60,
        label="Posterior",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
    frameon=True,
    fontsize=10,
)

## Decode the posterior

#### Model loading

In [ ]:
import torch
from torch.utils.data import DataLoader

from modules.model import VariationalAutoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset

checkpoint = torch.load(output_dir / 'model.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
device = torch.device('cpu')
latent_dim = checkpoint['LATENT_DIM']

#### Decoding

In [ ]:
model.eval()

with torch.no_grad():
    z = torch.from_numpy(Z_analysis).float().to(device)
    x_analysis = model.decode(z)
    x_analysis_raw = transform.invert(x_analysis).squeeze()
    
    # Detach and move to CPU once right after creation
    x_analysis_raw = x_analysis_raw.detach().cpu()

In [ ]:
posterior_samples = xr.Dataset(
    data_vars={
        "samples": (
            ("ens", "lat", "lon"),
            x_analysis_raw.numpy()
        ),
        "mean": (
            ("lat", "lon"),
            x_analysis_raw.mean(dim=0).numpy()
        ),
    },
    coords={
        "ens": np.arange(nens),
        #"lat": lat,
        #"lon": lon,
    },
)

posterior_samples.to_netcdf(output_dir / "posterior-samples.nc")